# AI Security & Governance — Hands-On

Offline lab: PII redaction, RAG RBAC, audit logs, content filtering, and a simple enterprise risk checklist.

## 0. Setup

In [ ]:
%pip install -q numpy
import re, hashlib, json, numpy as np
from datetime import datetime

def sha(x): return hashlib.sha256(x.encode()).hexdigest()[:8]

## 1. PII detection and deterministic pseudonymization

In [ ]:
PII_PATTERNS = {
    "email": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    "phone": re.compile(r"\b(?:\+?1[-. ]?)?\(?\d{3}\)?[-. ]?\d{3}[-. ]?\d{4}\b"),
    "ssn": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
}
def redact(text):
    findings=[]
    for kind, pat in PII_PATTERNS.items():
        def repl(m):
            findings.append((kind, m.group(0)))
            return f"<{kind.upper()}:{sha(m.group(0))}>"
        text = pat.sub(repl, text)
    return text, findings
redacted, findings = redact("jane@corp.com 415-555-1212 123-45-6789")
print(redacted, findings)
assert len(findings) == 3

## 2. RBAC and tenant boundaries before retrieval

In [ ]:
class AuditLog:
    def __init__(self): self.events=[]
    def record(self, **kw): self.events.append(kw)

def can_retrieve(user, doc, log):
    ok = user["tenant"] == doc["tenant"] and user["role"] in doc["roles"]
    log.record(user=user["id"], doc=doc["id"], decision="allow" if ok else "deny")
    return ok
log=AuditLog(); user={"id":"u7","role":"analyst","tenant":"acme"}
docs=[{"id":"policy","roles":{"analyst"},"tenant":"acme"},{"id":"payroll","roles":{"hr"},"tenant":"acme"}]
print([d["id"] for d in docs if can_retrieve(user,d,log)])
print(log.events)
assert len(log.events)==2

## 3. Content filtering for data leakage prevention

In [ ]:
BLOCK = [re.compile(r"api[_-]?key\s*=", re.I), re.compile(r"BEGIN RSA PRIVATE KEY"), re.compile(r"password\s*[:=]", re.I)]
def leakage_filter(text): return any(p.search(text) for p in BLOCK)
for out in ["safe answer", "password: secret"]:
    print(out, "->", "block" if leakage_filter(out) else "allow")
assert leakage_filter("api_key=abc")

## 4. Enterprise AI risk checklist scoring

In [ ]:
CHECKS = {"pii_minimized":2, "rbac":2, "audit_log":1, "evals":1, "human_escalation":1, "vendor_dpa":1}
def risk_score(controls):
    missing = [k for k in CHECKS if not controls.get(k)]
    return sum(CHECKS[k] for k in missing), missing
score, missing = risk_score({"pii_minimized":True,"rbac":True,"audit_log":False,"evals":True})
print("residual risk", score, "missing", missing)
assert "audit_log" in missing

## 5. Exercises and links
1. Add a region field and deny cross-region retrieval.
2. Add GDPR erasure logging.
3. Add severity levels to leakage filters.

Links: OWASP LLM Top 10, NIST AI RMF, EU AI Act.